# Multi-Layer Perceptron Network Demo

This is a minimal demo of how you can implement a basic MLP network in PyTorch, and train it for a simple image classification task.

### How to use this demo (advice)

First, skim this file, reading the code and the comments to get the general structure. This is intended to demonstrate everything we've seen so far in the course. Then, have a deeper look and try to actually run the code and play around with it.

**Note:** You cannot play around with this demo directly. You *could* make a copy for yourself (go to the "File" menu and choose "Save a copy in Drive", then use your copy). **However** I recommend you don't do that; instead: open another tab, create a new notebook from scratch, then one code cell at a time *read* the demo, try to memorize it, and *re-type* it in your notebook (maybe do it a piece at a time for longer cells). All of it, even the most trivial parts; *don't use copy-and-paste*. This is a trick that helps memorization and understanding, and forces you to pay attention to all the details (make sure that you understand the details that you don't get: read PyTorch's docs, use Google, ask ChatGPT or Gemini or Claude or a colleague or your grandma, etc.). After that, also play around with the network architecture (change the number and size of the layers, insert normalization/regularization layers, ...), play around with the parameters etc.

### Setup

Load the libraries

In [ ]:
import torch                                   # main module
from torch import nn                           # we'll use neural networks...
from torch.utils.data import DataLoader        # DataLoader is a convenient wrapper (see below)
from torchvision import datasets               # we'll use a "vision" dataset (for an image classification task)
from torchvision.transforms import ToTensor    # we'll need this to convert images into tensors

Prepare to use the GPU, if possible.

In general you can check whether you have a GPU (and of which type) with these functions:
* for NVIDIA cards: `torch.cuda.is_available()`
* for MacOS with Metal programming framework: `torch.backends.mps.is_available()`

In Google Colab, you can get a GPU like this:
* Go to the Edit menu -> Notebook Setting
* Select "Hardware accelerator" -> GPU
* Check the top-right "Resources", where there's RAM, Disk (you may need to connect or re-connect at this point); you can open the menu and select "View resources", and check if there's a GPU there

We'll create a string and put it in the `device` variable, so that later we can use methods such as `.to(device)` and pass our data and models to the appropriate device. If no GPU is available, this will fall back to using the CPU (i.e., do nothing).

In [ ]:
device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
print(f"Using {device}")

## The data

### Fetch the training and test data

We use the [FashionMNIST](https://github.com/zalandoresearch/fashion-mnist) task, a small image dataset with the same characteristics as the famous MNIST, but harder

The sizes are:
* training set size: $m = |S| = 60000$
* input size:        $N = 28*28 = 784$
* number of classes: $K = |Y| = 10$

In [ ]:
training_data = datasets.FashionMNIST(
    root = "data",
    train = True,
    download = True,
    transform = ToTensor(), # we need this to feed images into our models
)

test_data = datasets.FashionMNIST(
    root = "data",
    train = False,
    download = True,
    transform = ToTensor(),
)

Let's inspect what `training_data` and `test_data` actually are. They are `torch.utils.data.Dataset` objects — indexable collections of samples. Indexing with `[i]` returns a `(image_tensor, label)` tuple. No batching yet; that is the DataLoader's job.

In [ ]:
print(f"Type:          {type(training_data)}")
print(f"Training set:  {len(training_data)} samples")
print(f"Test set:      {len(test_data)} samples")
print()

## Indexing returns a (image_tensor, label) tuple — a single sample, no batch dimension
x, y = training_data[0]
print(f"x type:   {type(x)}")
print(f"x shape:  {x.shape}   — (C, H, W): 1 channel (grayscale), 28×28 pixels")
print(f"x dtype:  {x.dtype}   — ToTensor() converted uint8 [0,255] → float32 [0,1]")
print(f"x range:  [{x.min():.3f}, {x.max():.3f}]")
print()
print(f"y type:   {type(y)}")
print(f"y value:  {y}         — a plain integer, the class index (0–9)")

### Prepare the data loaders

We want to use minibatches:

1. Set the (mini-)batch size

2. Wrap the data into a `DataLoader` object. This creates an iterable that produces minibatches: at every iteration it gives us a tuple `(X,Y)`, see below.

Note: For testing, the only reason for using minibatches is that the full dataset may not fit into memory.

In [ ]:
batch_size = 128

train_dataloader = DataLoader(training_data, batch_size = batch_size, shuffle = True)
test_dataloader = DataLoader(test_data, batch_size = batch_size)

Notice that `shuffle=True` is set for the training loader but not for the test loader. Here is why it matters.

At the beginning of each epoch the DataLoader decides in what order to serve the samples. Without shuffling it always iterates through the dataset in the same fixed order — the order in which the samples happen to be stored on disk. This is problematic for two reasons:

* **Correlated mini-batches.** If the dataset is sorted by class (all T-shirts first, then all trousers, ...), every mini-batch will contain samples from only one or a few classes. The gradient computed from such a batch gives a very biased estimate of the true gradient over the whole dataset, making training noisy and slow.
* **The model can exploit the order.** In pathological cases the network may start to "memorise" the sequence of batches rather than learning the underlying patterns.

Shuffling at the start of each epoch randomises the order, so every mini-batch is a representative mixture of classes — giving a much better gradient estimate. This is the standard practice for training, and it has no downside.

For the test/validation loader, order is irrelevant (we are just evaluating, not updating weights), so there is no reason to shuffle, and skipping it makes the results slightly more reproducible.

Here is a small demo of a `DataLoader` object. We look at the first minibatch and check the dimensions of the data in it. Note: we use a for loop but break immediately: this is necessary because `DataLoader` is an iterable but it's not indexable, we can't just call `test_dataloader[0]`, thus we need to trigger the iteration protocol and then stop right away.

In [ ]:
for X,y in test_dataloader:
    ## The X is a 4d tensor:
    ##   the first dimension is the pattern so it has size b=batch_size
    ##   then there is the channel, in our case the size is C=1 (the images are grayscale)
    ##   then there are rows and columns, with size HxW
    print(f"Shape of X [b, C, H, W]: {X.shape}")
    ## The y is a 1d tensor:
    ##   the dimension is the pattern, the size is b=batch_size
    ##   each entry is a label between 0 and 9
    print(f"Shape of y {y.shape} {y.dtype}")
    break

## The model

### Create the Neural Network

A network must inherit from `nn.Module`.

At the very minimum, we need the `__init__()` and `forward()` methods. The rest of the methods are inherited from `nn.Module`.

**Construction**

The `__init__` is where we put all the architectural elements of our network. For our simple case, we'll create a purely feed-forward sequential network. Thus, we can put everything inside a single attribute, a `nn.Sequential` object.

The `nn.Sequential` object is also a kind of `nn.Module`. Its only purpose is to create a sequence of modules; the input is given to the first module and then the output of each module is passed into the following one until the last one gives us the final output of the sequence.

Inside, we put the layers. All the layers defined in pytorch also inherit from `nn.Module`. Note that even `nn.ReLU` is an object (a ReLU layer); when we write `nn.ReLU()` we are creating an object, which will then behave like a function if we call it (it takes a tensor argument and returns a tensor of the same shape).

Thanks to inheritance, the optimizer (e.g. SGD) can know which parmeters it should update.

Our first operation must be to flatten the images, since MLPs operate on vectors (this will keep the mini-batch index and flatten the channel/width/height, so the output is 2d, not 1d).

The number of input features in the first layer is forced upon us by the data (here it's $N=28*28$).

The number of outputs in the last layer is the number of classes (here $K=10$).

The rest is arbitrary and up to us, but it must be consistent between one layer and the next.

**Forward pass**

Having put all the structure in a sequence, the definition of the forward pass is trivial. Note that the argument `x` here is a mini-batch, so for us it's a 4d tensor (pattern, channel, row, column).

Notice however that this mathod does not include the softmax, it outputs the logits (which is a 2d tensor with the pattern index first, followed by the class index).

In [ ]:
class NeuralNetwork(nn.Module):
    def __init__(self):
        ## Invoke the parent constructor to set up out object
        super().__init__()
        ## We'll only need one (composite) attribute
        self.net = nn.Sequential(
            nn.Flatten(),              # flatten the 4d tensors into 2d tensors
            nn.Linear(28*28, 512),     # a dense layer, reducing the size from 768 to 512
            nn.ReLU(),                 # nonlinearity, applies ReLU element-wise
            nn.Linear(512, 512),       # a dense layer, keeping the size fixed
            nn.ReLU(),
            nn.Linear(512, 10),        # output layer; note it's not followed by anything
            )

    def forward(self, x):
        logits = self.net(x)
        return logits

The network above uses `nn.Sequential`, which is convenient but hides the forward pass. Here is the exact same network written in the alternative style, where each layer is a named attribute and the forward pass is spelled out explicitly. This style is more verbose but gives you full control — you can add branches, skip connections, or any logic you like.

Notice that in the `Sequential` version we used `nn.ReLU()` (with parentheses), which creates a **module object**. In the explicit forward pass below we instead use `F.relu(...)`, a plain **function** imported from `torch.nn.functional`. The difference:

* `nn.ReLU()` is an `nn.Module` instance. It has no learnable parameters, but it *is* an object with state — it can be listed in `print(model)`, registered as a submodule, etc. This is the natural choice inside `nn.Sequential`.
* `F.relu` is a stateless function you call directly on a tensor inside `forward()`. For activation functions with no parameters (ReLU, sigmoid, tanh, ...) both are equivalent in terms of what they compute. The functional style is common when writing explicit `forward` methods because it keeps `__init__` clean: you only declare things that have *learnable parameters* there.

As a rule of thumb: layers with parameters (`nn.Linear`, `nn.Conv2d`, `nn.BatchNorm1d`, ...) must always be declared as attributes in `__init__` so PyTorch can find and optimise their parameters. Stateless operations (activations, dropout during inference, etc.) can go either way.

In [ ]:
import torch.nn.functional as F   ## the functional API: stateless versions of most nn operations

class NeuralNetworkExplicit(nn.Module):
    def __init__(self):
        super().__init__()
        ## Only layers WITH learnable parameters go here
        self.flatten = nn.Flatten()        ## no parameters, but convenient to keep here
        self.fc1 = nn.Linear(28*28, 512)   ## first hidden layer
        self.fc2 = nn.Linear(512,   512)   ## second hidden layer
        self.fc3 = nn.Linear(512,    10)   ## output layer

    def forward(self, x):
        ## The forward pass is now written out step by step
        x = self.flatten(x)        ## (B, 1, 28, 28) → (B, 784)
        x = F.relu(self.fc1(x))    ## linear → relu; F.relu is a function, not a module
        x = F.relu(self.fc2(x))    ## same for the second layer
        x = self.fc3(x)            ## no activation on the output: we return raw logits
        return x

## Both classes produce identical results — let's verify
model_seq      = NeuralNetwork().to(device)
model_explicit = NeuralNetworkExplicit().to(device)

dummy_input = torch.randn(4, 1, 28, 28).to(device)   ## fake batch of 4 images
out_seq      = model_seq(dummy_input)
out_explicit = model_explicit(dummy_input)

print(f"Sequential output shape:  {out_seq.shape}")
print(f"Explicit   output shape:  {out_explicit.shape}")
print()
print("Note: the outputs won't be numerically equal because the weights are randomly")
print("initialised independently — but the shapes and behaviour are identical.")
print()

## print() reveals one difference: Sequential lists ReLU as a submodule, explicit does not
print("--- Sequential model ---")
print(model_seq)
print()
print("--- Explicit model ---")
print(model_explicit)

### A note on weight initialisation

When you create an `nn.Linear` layer, PyTorch immediately fills its weights and biases with random values — it does not wait for you to call anything. The choice of *how* to initialise matters enormously: if weights start too large, activations and gradients explode through the layers; too small and they vanish.

By default, `nn.Linear` uses **Kaiming uniform** initialisation (also called He initialisation, after the author): weights are drawn from a uniform distribution whose range is chosen so that the variance of the activations stays roughly constant from one layer to the next, assuming ReLU activations. The biases are initialised with a similar uniform scheme. This is a sensible default for ReLU networks.

You will also encounter **Xavier / Glorot** initialisation, which targets the same goal but assumes symmetric activations (sigmoid, tanh). Both are available in `torch.nn.init` if you ever need to override the defaults.

You can inspect or overwrite the initial values at any time — the weights are just tensors stored as `.weight.data` and `.bias.data`.

In [ ]:
## Inspect the default initialisation of the first linear layer
layer = model_explicit.fc1
print(f"fc1 weight shape: {layer.weight.shape}")   ## (out_features, in_features) = (512, 784)
print(f"fc1 bias   shape: {layer.bias.shape}")     ## (out_features,) = (512,)
print()
print(f"weight — min: {layer.weight.data.min():.4f},  max: {layer.weight.data.max():.4f},  "
      f"std: {layer.weight.data.std():.4f}")
print(f"bias   — min: {layer.bias.data.min():.4f},  max: {layer.bias.data.max():.4f},  "
      f"std: {layer.bias.data.std():.4f}")
print()

## You can re-initialise with any scheme from torch.nn.init
torch.nn.init.xavier_uniform_(layer.weight)   ## Xavier/Glorot: good for sigmoid/tanh
torch.nn.init.zeros_(layer.bias)              ## biases to zero is also common
print("After xavier_uniform_ + zeros_:")
print(f"weight — min: {layer.weight.data.min():.4f},  max: {layer.weight.data.max():.4f},  "
      f"std: {layer.weight.data.std():.4f}")
print(f"bias   — all zero: {layer.bias.data.abs().max().item() == 0.0}")

Now we create a network and send it to the device we'll be using.
* If that is the CPU, nothing will happen.
* If it's a GPU, data will be copied over to the GPU memory.

In [ ]:
model = NeuralNetwork().to(device)

We can get an overview of the model

In [ ]:
print(model)

### Loss function

We now choose a loss function. We'll use the most standard one. Note that what is called "cross entropy" here is the combination of softmax and crossentropy.

**Note:** we *call* a function `CrossEntopyLoss` that creates an object `loss_fn` that we will *then* call as if it were a function.

In [ ]:
loss_fn = nn.CrossEntropyLoss()

### Optimizer

Next, we choose an optimizer. Note that the `parameters()` method knows which tensors to fetch from our model, because our model is a `nn.Module` which in turn is a composition of `nn.Module` objects. These are the parameters that will be optimized.

In [ ]:
optimizer = torch.optim.SGD(model.parameters(), lr=1e-2)
# optimizer = torch.optim.Adam(model.parameters())

## Train it and use it

### Train and test functions

Here we define a training function. This performs a single training epoch (1 pass thorugh the training set). It loads minibatches one at a time, and for each one it does the forward pass (computing the output and all the intermediate values, and registering the computational graph), the backward pass (walking down the computational graph and storing the gradients), the optimization step (applying the gradients).

It will also collect some statistics on the training loss and training error as it proceeds; note however that these are computed on each minibatch right before the network changes, so they won't be accurate (especially at the start). Use them just as a rough guide.

Two other important details:
* we set the network in `train` mode (important for batchnorm and dropout layers)
* we send the training data to the same device where the model is stored (so, the GPU if available).

In [ ]:
def train(dataloader, model, loss_fn, optimizer):
    size = len(dataloader.dataset) # the .dataset attribute gives us the original data, so this is 60000
    num_batches = len(dataloader)  # the dataloader is an iterable that produces batchs, so its length is the num. of batches
    model.train()                  # set the model in training mode (important for the batchnorm and dropout layers)
    mean_loss = 0.0
    correct = 0
    for batch, (X, y) in enumerate(dataloader):
        X, y = X.to(device), y.to(device)  # send the data to the device. This is crucial if we're using the GPU

        ## Forward pass.
        ## This computes the loss but it also builds the computational graph,
        ##   storing it in a distributed fashion in the various tensors involved.
        ## The loss is a tensor with a single entry, not a float. This is crucial for backpropagation.
        logits = model(X)          # pass thorugh the model; get the logits
        loss = loss_fn(logits, y)  # compute the loss; note that `y` will be translated into a 1-hot-encoded vactor

        ## Backward pass.
        optimizer.zero_grad() # reset the gradients inside all tensors that were given as parameters to the optimizer
        loss.backward()       # trigger the backward pass; here all the gradients are computed; each tensor stores its own
        optimizer.step()      # apply one step of optimization to the parameters, using the gradient information

        ## Here we just keep track of the loss and error to monitor the training.
        ## Note that the `item()` method gives you the value of a single-valued tensor, such as the loss.
        mean_loss += loss.item()
        correct += (logits.argmax(1) == y).type(torch.float).sum().item()

        ## Some crude visual feedback to display our progress
        if batch % 100 == 0:
            loss, current = loss.item(), (batch+1) * len(X)
            print(f"  [{current:>5d}/{size:>5d}]")

    ## Compute and print some data to monitor the training
    mean_loss /= num_batches
    correct /= size
    print(f"TRAINING - Accuracy: {(100 * correct):>5.1f}%, Avg loss: {loss:>7f}")

You might wonder: why do we call `X, y = X.to(device), y.to(device)` *inside* the loop, on every single batch, rather than moving the entire dataset to the GPU once before training starts?

The answer is **memory**. A GPU typically has far less memory than RAM (8–24 GB on a consumer card vs. 32–128 GB of RAM). FashionMNIST is small enough that it would fit, but a real-world dataset — millions of high-resolution images, hours of audio, gigabytes of text — will not. The standard pattern is therefore to keep the full dataset in CPU RAM, and move each mini-batch to the GPU just before it is needed. The batch is processed, the GPU memory is freed, and the next batch is moved over.

This means the `.to(device)` call happens on a small tensor (one batch) rather than on the entire dataset, and the overhead is negligible compared to the GPU computation that follows. The model itself, however, is moved to the GPU once (`model = NeuralNetwork().to(device)`) and stays there for the entire training run — its weights are reused across all batches so there is no reason to move them back and forth.

We also define a test function. No optimization going on here, so we set the model in eval mode and use `torch.no_grad()` to skip the computational graph computation. Apart from that, the rest is very similar to the training.

In [ ]:
def test(dataloader, model, loss_fn):
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    model.eval()                   # set the model in evaluation mode (important for the batchnorm and dropout layers)
    test_loss, correct = 0.0, 0
    ## We operate in a `no_grad` mode; this is faster since it avoids building the computational graph during the forward pass
    with torch.no_grad():
        for X, y in dataloader:
            X, y = X.to(device), y.to(device)
            logits = model(X)
            test_loss += loss_fn(logits, y).item()
            correct += (logits.argmax(1) == y).type(torch.float).sum().item()
    test_loss /= num_batches
    correct /= size
    print(f"TESTING  - Accuracy: {(100 * correct):>5.1f}%, Avg loss: {test_loss:>8f}\n")

### Train the network

Finally, we're ready for the training. We set up a number of epochs and go. If we're not happy, we can keep going on for more epochs, change the learning rate or even the optimization method, etc.

In [ ]:
epochs = 2
for t in range(epochs):
    print(f"Epoch {t+1}\n------------------")
    train(train_dataloader, model, loss_fn, optimizer)
    test(test_dataloader, model, loss_fn)
print("Done!")

### Use the model after training

Here we demonstrate how we could use this model to actually predict the class of an image.

The first thing we need are the association between the label indices and their meaning; in other words we need to tranlsate the numbers 0-9 into strings.

In [ ]:
classes = [
    "T-shirt/top",
    "Trouser",
    "Pullover",
    "Dress",
    "Coat",
    "Sandal",
    "Shirt",
    "Sneaker",
    "Bag",
    "Ankle boot",
]

Let's pick up a test sample for the demo.

We're using the test data here, but of course in a realistic scenario you would acquire an image, turn it into a tensor, possibly do some pre-processing (in case you did it for the train; we didn't but we could have). Then you'd sent it to the appropriate device.

Note that now `x` is 3d and `y` is just a number (we're not using the data loaders and we're not getting minibatches).

In [ ]:
x, y = test_data[0][0].to(device), test_data[0][1]

We set the model in evaluation mode, like in the test function above; we'll also use `torch.no_grad`.

Then we predict the class (and in this case we can compare it with the correct one). Note that again we don't have minibatches, so the output now is 1d instead of 2d.

In [ ]:
model.eval()
with torch.no_grad():
    logits = model(x) # returns a 1d tensor; its argmax is the predicted class index
    predicted, actual = classes[logits[0].argmax(0)], classes[y]
    print(f"Predicted: '{predicted}', Actual: '{actual}'")
